---
# Методы статистической обработки информации
---
## Лабораторная работа 2 / Регрессионный анализ / Вариант 15
---

---
## Подготовка
---

### Импорт библиотек

In [186]:
from math             import log10
from MoSDP_data       import get_data
from MoSDP_operations import *

### Получение данных варианта

In [187]:
variant  = 15 # int(input("Введите номер варианта: "))
var_data = get_data(variant)

### Подготовка данных


In [188]:
Y_linear = transpose([var_data[0]])
X_linear = transpose(var_data[1:])

n = len(X_linear)

for x in X_linear:
    x.insert(0, 1)

numbers_linear = [i for i in range(len(X_linear[0]))]
numbers_pow    = [i for i in range(len(X_linear[0]))]

### Приложение

In [189]:
# По "таблице 2": a = 0.05 и v = n - k - 1
t_table = {(0.05, 40): 2.021, (0.1, 40): 1.684, (0.2, 40): 1.303, (0.3, 40): 1.050 , (0.4, 40): 0.851}

# По "таблице 4": a = 0.05; v1 = k + 1 и v2 = n - k - 1
F_table = {(0.05, 6, 47): 2.25, (0.05, 5, 48): 2.37, (0.05, 4, 49): 2.52, (0.05, 3, 50): 2.76, (0.05, 2, 51): 3.15}

---
## Задание 1
---

### Функция "Оценка уравнения регрессии"

In [190]:
def estimate(Y, X, form):
    xTx = matrix_multiplication(transpose(X), X)
    xTy = matrix_multiplication(transpose(X), Y)
    b   = matrix_multiplication(inverse_matrix(xTx), xTy)

    if form == "pow":
        b[0][0] = 10 ** b[0][0]

    return b

### Функция "Формирование уравнения"

In [191]:
def draw(b, numbers, form):
    equation = f"y = {b[0][0]}"

    for i in range(1, len(b)):
        if form == "linear":
            if b[i][0] > 0:
                equation += f" + {b[i][0]}"
            else:
                equation += f" - {abs(b[i][0])}"
            equation += f" * x{numbers[i]}"
        else:
            equation += f" * x{numbers[i]}^({b[i][0]})"

    print(equation)

### Оценка линейного уравнения регрессии

In [192]:
b_linear = estimate(Y_linear, X_linear, "linear")
print("b_linear:", b_linear, "\n")

draw(b_linear, numbers_linear, "linear")

b_linear: [[-12.560998303339147], [18.21127665232234], [1.6273983943464145], [0.5084514879716515], [0.2785008638589499], [0.21507667359831828]] 

y = -12.560998303339147 + 18.21127665232234 * x1 + 1.6273983943464145 * x2 + 0.5084514879716515 * x3 + 0.2785008638589499 * x4 + 0.21507667359831828 * x5


### Оценка степенного уравнения регрессии

In [193]:
Y_pow = copy(Y_linear)
X_pow = copy(X_linear)

for i in range(len(X_linear)):
    Y_pow[i][0] = log10(Y_pow[i][0])

    for j in range(1, len(X_linear[0])):
        X_pow[i][j] = log10(X_pow[i][j])

b_pow = estimate(Y_pow, X_pow, "pow")
print("b_pow:", b_pow, "\n")

draw(b_pow, numbers_pow, "pow")

b_pow: [[0.5749714020371827], [3.591015023519958], [-0.027089077716006216], [0.24718836073139272], [0.2095603646767561], [1.0877541678012221]] 

y = 0.5749714020371827 * x1^(3.591015023519958) * x2^(-0.027089077716006216) * x3^(0.24718836073139272) * x4^(0.2095603646767561) * x5^(1.0877541678012221)


---
## Задание 2
---

### Функция "Значимость уравнения"

In [194]:
def significance_equation(Y, X, b, form):
    b_current = copy(b)

    if form == "pow":
        b_current[0][0] = log10(b_current[0][0])

    Y_reg = matrix_multiplication(X, b_current)

    Q = 0
    for i in range(len(Y)):
        Q += (Y[i][0] - Y_reg[i][0]) ** 2

    k   = len(X[0]) - 1
    S_2 = Q / (n - k - 1)

    Q_r = matrix_multiplication(transpose(Y_reg), Y_reg)

    F_critical = F_table[(0.05, k + 1, n - k - 1)]
    F          = (Q_r[0][0] / (k + 1)) / S_2

    res = ('<=', 'не ')
    if F > F_critical:
        res = ('>', '')

    print(f"F_набл = {F} {res[0]} {F_critical}: {res[1]}значимое")

    return S_2

### Значимость линейного уравнения

In [195]:
S_2_linear = significance_equation(Y_linear, X_linear, b_linear, "linear")

F_набл = 83.74135090633733 > 2.25: значимое


### Значимость степенного уравнения

In [196]:
S_2_pow = significance_equation(Y_pow, X_pow, b_pow, "pow")

F_набл = 434.62842715579666 > 2.25: значимое


### Функция "Значимость коэффициентов регрессии"

In [197]:
def significance_coefficients(X, b, S_2, t_critical):
    xTx = inverse_matrix(matrix_multiplication(transpose(X), X))
    S   = matrix_by_a_number(xTx, S_2)

    t = []
    for i in range(1, len(S)):
        t.append(b[i][0] / (S[i][i] ** 0.5))

        res = (' <=', 'не')
        if abs(t[-1]) > t_critical:
            res = (' > ', '  ')

        print(f"t_набл(b{i}) = |{t[-1]:{7}.3f}| {res[0]} {t_critical}: {res[1]} значимый")

        t[-1] = abs(t[-1])

    return t

### Значимость коэффициентов регрессии линейного уравнения

In [198]:
t_linear = significance_coefficients(X_linear, b_linear, S_2_linear, t_table[(0.05, 40)])

t_набл(b1) = |  0.860|  <= 2.021: не значимый
t_набл(b2) = |  0.598|  <= 2.021: не значимый
t_набл(b3) = |  0.428|  <= 2.021: не значимый
t_набл(b4) = |  1.457|  <= 2.021: не значимый
t_набл(b5) = |  0.906|  <= 2.021: не значимый


### Значимость коэффициентов регрессии степенного уравнения

In [199]:
t_pow = significance_coefficients(X_pow, b_pow, S_2_pow, t_table[(0.05, 40)])

t_набл(b1) = |  2.524|  >  2.021:    значимый
t_набл(b2) = | -0.438|  <= 2.021: не значимый
t_набл(b3) = |  1.420|  <= 2.021: не значимый
t_набл(b4) = |  1.818|  <= 2.021: не значимый
t_набл(b5) = |  2.571|  >  2.021:    значимый


---
## Задание 3
---

### Функция "Мультиколлинеарность"

In [200]:
def multicollinearity(X, numbers):
    print("Матрица парных коэффициентов корреляции:")
    R = pairwise_correlation_coefficients(X)
    print()

    related = []
    for i in range(1, len(R)):
        for j in range(i + 1, len(R)):
            if abs(R[i][j]) > 0.8:
                print(f"X{i} и X{j} - тесно связанные: r_{i}{j} = |{R[i][j]:.3f}| > 0.8")

                related.append(i)
                related.append(j)

    X_without_related    = copy(X)
    X_without_related[0] = [1] * len(X_without_related[0])

    numbers_without_related = numbers.copy()

    if len(related) > 0:
        i = list(set(related))[0]

        max_correlation = abs(R[0][i])
        for j in related[1:]:
            if max_correlation < abs(R[0][j]):
                max_correlation = abs(R[0][j])
                i               = j

        print(f"\nОставляем X{i} из X{", X".join(str(j) for j in related)}")
        related.remove(i)

        for j in related:
            X_without_related.pop(j)
            numbers_without_related.pop(j)

    else:
        print("Нет тесно связанных X")

    return transpose(X_without_related), numbers_without_related

### Мультиколлинеарность линейного уравнения

In [201]:
X_without_related_linear, numbers_without_related_linear = multicollinearity(var_data, numbers_linear)

Матрица парных коэффициентов корреляции:
[1                        | 0.054840792487636555     | 0.15115051013848507      | 0.0003555860122828476    | 0.23405035667830304      | 0.018075047394182264     ]
[0.054840792487636555     | 1                        | -0.06874216425985696     | -0.016615678293707227    | 0.2193289370025393       | -0.9402329788072865      ]
[0.15115051013848507      | -0.06874216425985696     | 1                        | 0.4333027759567459       | -0.04467957562887469     | 0.14675271575071883      ]
[0.0003555860122828476    | -0.016615678293707227    | 0.4333027759567459       | 1                        | -0.5084487611839539      | 0.03736158062663443      ]
[0.23405035667830304      | 0.2193289370025393       | -0.04467957562887469     | -0.5084487611839539      | 1                        | -0.14710495223940964     ]
[0.018075047394182264     | -0.9402329788072865      | 0.14675271575071883      | 0.03736158062663443      | -0.14710495223940964     | 1       

### Мультиколлинеарность степенного уравнения

In [202]:
var_data_lg = copy(var_data)

for row in var_data_lg:
    for i in range(len(row)):
        row[i] = log10(row[i])

X_without_related_pow, numbers_without_related_pow = multicollinearity(var_data_lg, numbers_pow)

Матрица парных коэффициентов корреляции:
[1                        | 0.10259764042570861      | 0.1327619563703737       | 0.07812534391931236      | 0.21797415230395475      | 0.05768242341919306      ]
[0.10259764042570861      | 1                        | -0.09311330813646256     | -0.006143617306348758    | 0.19597908620142443      | -0.9100093133233252      ]
[0.1327619563703737       | -0.09311330813646256     | 1                        | 0.4025979614542848       | -0.0912299433840107      | 0.2406051574322556       ]
[0.07812534391931236      | -0.006143617306348758    | 0.4025979614542848       | 1                        | -0.4446836830988275      | 0.002790920878960479     ]
[0.21797415230395475      | 0.19597908620142443      | -0.0912299433840107      | -0.4446836830988275      | 1                        | -0.14163967790330317     ]
[0.05768242341919306      | -0.9100093133233252      | 0.2406051574322556       | 0.002790920878960479     | -0.14163967790330317     | 1       

### Функция "Алгоритм пошагового регрессионного анализа"

In [203]:
def regression_analysis(Y, X, numbers, form):
    for parameters in t_table:
        t_critical = t_table[parameters]

        print("=" * 97)
        print(f"При a = {parameters[0]}")
        print("=" * 97)

        new_X       = copy(X)
        new_numbers = numbers.copy()

        t = [0]

        step = 0
        while min(t) <= t_critical and len(t) > 1 or step == 0:
            if step != 0:
                print(f"{"-" * 45} Шаг {step} {"-" * 45}")

                j = t.index(min(t)) + 1
                print(f"\nМинимальный t: |t_набл(b{j})| <= {t_critical}")

                for x in new_X:
                    x.pop(j)

                new_numbers.pop(j)

            b = estimate(Y, new_X, form)
            print(f"\nНовая оценка b_{form}: {b}")

            draw(b, new_numbers, form)
            print()

            S_2 = significance_equation(Y, new_X, b, form)
            t   = significance_coefficients(new_X, b, S_2, t_critical)
            print()

            step += 1

        if min(t) > t_critical:
            print("=" * 97)
            print(f"Итоговая оценка b_{form}: {b}")
            print()
            draw(b, new_numbers, form)
            print("=" * 97)

            return new_X, b, new_numbers

    return None, None, None

### Пошаговый регрессионный анализ линейного уравнения

In [204]:
new_X_linear, b_linear, new_numbers_linear = regression_analysis(Y_linear, X_without_related_linear, numbers_without_related_linear, "linear")

При a = 0.05

Новая оценка b_linear: [[4.494981175979063], [0.10878043325672593], [2.0898857804358713], [0.5881921959285847], [0.3130236898524039]]
y = 4.494981175979063 + 0.10878043325672593 * x1 + 2.0898857804358713 * x2 + 0.5881921959285847 * x3 + 0.3130236898524039 * x4

F_набл = 100.69931482122335 > 2.37: значимое
t_набл(b1) = |  0.015|  <= 2.021: не значимый
t_набл(b2) = |  0.784|  <= 2.021: не значимый
t_набл(b3) = |  0.497|  <= 2.021: не значимый
t_набл(b4) = |  1.674|  <= 2.021: не значимый

--------------------------------------------- Шаг 1 ---------------------------------------------

Минимальный t: |t_набл(b1)| <= 2.021

Новая оценка b_linear: [[4.567481156384787], [2.0844885515195415], [0.5911472358960594], [0.3137978031043662]]
y = 4.567481156384787 + 2.0844885515195415 * x2 + 0.5911472358960594 * x3 + 0.3137978031043662 * x4

F_набл = 128.49582185580783 > 2.52: значимое
t_набл(b1) = |  0.797|  <= 2.021: не значимый
t_набл(b2) = |  0.512|  <= 2.021: не значимый
t_набл(b

### Пошаговый регрессионный анализ степенного уравнения

In [205]:
new_X_pow, b_pow, new_numbers_pow = regression_analysis(Y_pow, X_without_related_pow, numbers_without_related_pow, "pow")

При a = 0.05

Новая оценка b_pow: [[5.483016238452722], [0.2344293063718368], [0.03780886630689606], [0.18173918799332967], [0.22364587756914034]]
y = 5.483016238452722 * x1^(0.2344293063718368) * x2^(0.03780886630689606) * x3^(0.18173918799332967) * x4^(0.22364587756914034)

F_набл = 465.7814753274249 > 2.37: значимое
t_набл(b1) = |  0.392|  <= 2.021: не значимый
t_набл(b2) = |  0.634|  <= 2.021: не значимый
t_набл(b3) = |  0.999|  <= 2.021: не значимый
t_набл(b4) = |  1.838|  <= 2.021: не значимый

--------------------------------------------- Шаг 1 ---------------------------------------------

Минимальный t: |t_набл(b1)| <= 2.021

Новая оценка b_pow: [[4.965378364355461], [0.03486789963322012], [0.19140812739097868], [0.23456181540272425]]
y = 4.965378364355461 * x2^(0.03486789963322012) * x3^(0.19140812739097868) * x4^(0.23456181540272425)

F_набл = 592.4201621734647 > 2.52: значимое
t_набл(b1) = |  0.595|  <= 2.021: не значимый
t_набл(b2) = |  1.071|  <= 2.021: не значимый
t_набл

---
## Задание 4
---